### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "assets.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp06/assets.csv"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp06" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP06 — Filtering

Εφάρμοσε attribute και geometry-aware φίλτρα στο dataset υποδομών. Πρώτα πρόβλεψε το πλήθος, μετά εφάρμοσε Boolean mask και τέλος έλεγξε before/after, conditions, CRS, geometry και bounds.

Θυμήσου: `gdf['score'] > 50` είναι σειρά True/False· το filtered GeoDataFrame προκύπτει όταν χρησιμοποιήσεις αυτή τη μάσκα μέσα σε `gdf[...]`.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
df = pd.read_csv(DATA_ROOT / "gp06/assets.csv")
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326')
gdf.head()


In [ ]:
active = gdf[gdf['status'] == '____']
active


In [ ]:
selected = gdf[(gdf['status'] == 'active') & (gdf['score'] >= ____)]
selected


In [ ]:
types = gdf[gdf['asset_type'].isin([____, ____])]
types


In [ ]:
bbox = gdf.cx[23.55:23.60, 41.08:41.11]
bbox


Κάνε QA: μέτρα γραμμές, έλεγξε τις συνθήκες και επιβεβαίωσε ότι το CRS και οι γεωμετρίες παρέμειναν σωστά. Αν πάρεις 0 rows, έλεγξε spelling, case, missing values και thresholds πριν θεωρήσεις ότι δεν υπάρχουν matches.

In [ ]:
# assert ...
